In [5]:
!pip install requests beautifulsoup4 trafilatura pandas

In [6]:
import requests
from bs4 import BeautifulSoup
import trafilatura
import pandas as pd

print("Semua library berhasil di-import!")

Semua library berhasil di-import!


In [7]:
url = "https://sport.detik.com/"

response = requests.get(
    url,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("Status code:", response.status_code)
print("Panjang halaman:", len(response.text))

Status code: 200
Panjang halaman: 276914


In [9]:
import requests
from bs4 import BeautifulSoup

url = "https://sport.detik.com/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print("Status:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print("Soup berhasil dibuat")

Status: 200
Soup berhasil dibuat


In [10]:
from urllib.parse import urljoin, urlparse

links_sport = []

for a in soup.find_all("a", href=True):
    href = urljoin("https://sport.detik.com/", a["href"])
    parsed = urlparse(href)

    # Hanya mengambil link yang berasal dari sport.detik.com
    if parsed.netloc == "sport.detik.com":
        path = parsed.path

        # Hindari halaman utama, indeks, kategori, dan halaman navigasi
        if (
            path not in ["/", "/indeks", "/index", "/index/"]
            and "/indeks" not in path
            and "/index" not in path
            and not path.startswith("/tag/")
        ):
            links_sport.append(href)

# Hilangkan duplikat
links_sport = list(dict.fromkeys(links_sport))

print("Jumlah link ditemukan:", len(links_sport))

for link in links_sport[:10]:
    print(link)

Jumlah link ditemukan: 67
https://sport.detik.com/sepakbola
https://sport.detik.com
https://sport.detik.com/moto-gp
https://sport.detik.com/raket
https://sport.detik.com/f1
https://sport.detik.com/basket
https://sport.detik.com/sportstyle
https://sport.detik.com/sport-lain
https://sport.detik.com/foto
https://sport.detik.com/video


In [11]:
links_sport = []

for a in soup.find_all("a", href=True):
    href = urljoin("https://sport.detik.com/", a["href"])

    # Artikel Detik biasanya memiliki /d- pada URL
    if "sport.detik.com" in href and "/d-" in href:
        links_sport.append(href)

# Hilangkan duplikat
links_sport = list(dict.fromkeys(links_sport))

print("Jumlah link artikel Sport:", len(links_sport))

for link in links_sport[:10]:
    print(link)

Jumlah link artikel Sport: 51
https://sport.detik.com/moto-gp/d-8663795/marc-marquez-akui-dapat-kado-manis-dari-rival-rivalnya
https://sport.detik.com/sport-lain/d-8664208/eko-yuli-masih-incar-tiket-olimpiade-la-2028
https://sport.detik.com/moto-gp/d-8664759/mau-kompetitif-lagi-kenapa-quartararo-tak-coba-dapatkan-motor-ducati
https://sport.detik.com/raket/d-8665053/ana-trias-fokus-medali-di-asian-games-2026
https://sport.detik.com/moto-gp/d-8665387/senna-agius-lengkapi-grid-motogp-2027-simak-daftar-lengkap-rider-musim-depan
https://sport.detik.com/fotosport/d-8665222/serunya-549-siswa-di-ciamis-adu-ketangkasan-lomba-olahraga-tradisional
https://sport.detik.com/sepakbola/liga-spanyol/d-8665696/nilai-7-untuk-emil-audero
https://sport.detik.com/sepakbola/liga-indonesia/d-8665937/sikat-fc-seoul-persib-bandung-bikin-sejarah-di-negeri-gingseng
https://sport.detik.com/sepakbola/liga-indonesia/d-8665639/indonesia-vs-thailand-difavoritkan-jadi-final-fifa-asean-cup-2026
https://sport.detik.com/s

In [12]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

def ambil_link_sport(target=100):
    links = []

    for halaman in range(1, 10):
        url = f"https://sport.detik.com/indeks?page={halaman}"

        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        print(f"Mengambil halaman {halaman} - Status: {response.status_code}")

        soup = BeautifulSoup(response.text, "html.parser")

        for a in soup.find_all("a", href=True):
            href = urljoin("https://sport.detik.com/", a["href"])

            if "sport.detik.com" in href and "/d-" in href:
                links.append(href)

        # Hilangkan duplikat
        links = list(dict.fromkeys(links))

        print(f"Total link unik: {len(links)}")

        if len(links) >= target:
            break

        time.sleep(1)

    return links[:target]


links_sport = ambil_link_sport(100)

print("\nJumlah link Sport yang dikumpulkan:", len(links_sport))

Mengambil halaman 1 - Status: 200
Total link unik: 20
Mengambil halaman 2 - Status: 200
Total link unik: 39
Mengambil halaman 3 - Status: 200
Total link unik: 57
Mengambil halaman 4 - Status: 200
Total link unik: 77
Mengambil halaman 5 - Status: 200
Total link unik: 97
Mengambil halaman 6 - Status: 200
Total link unik: 117

Jumlah link Sport yang dikumpulkan: 100


In [13]:
import trafilatura
import time

data_sport = []

for i, url in enumerate(links_sport, start=1):
    try:
        downloaded = trafilatura.fetch_url(url)
        isi = trafilatura.extract(downloaded)

        if isi:
            data_sport.append({
                "isi_berita": isi,
                "label": "sport"
            })

            print(f"{i}/100 berhasil")
        else:
            print(f"{i}/100 gagal mengambil isi")

    except Exception as e:
        print(f"{i}/100 error: {e}")

    time.sleep(1)

print("\nTotal berita Sport berhasil:", len(data_sport))

1/100 berhasil
2/100 berhasil
3/100 berhasil
4/100 berhasil
5/100 berhasil
6/100 berhasil
7/100 berhasil
8/100 berhasil
9/100 berhasil
10/100 berhasil
11/100 berhasil
12/100 berhasil
13/100 berhasil
14/100 berhasil
15/100 berhasil
16/100 berhasil
17/100 berhasil
18/100 berhasil
19/100 berhasil
20/100 berhasil
21/100 berhasil
22/100 berhasil
23/100 berhasil
24/100 berhasil
25/100 berhasil
26/100 berhasil
27/100 berhasil
28/100 berhasil
29/100 berhasil
30/100 berhasil
31/100 berhasil
32/100 berhasil
33/100 berhasil
34/100 berhasil
35/100 berhasil
36/100 berhasil
37/100 berhasil
38/100 berhasil
39/100 berhasil
40/100 berhasil
41/100 berhasil
42/100 berhasil
43/100 berhasil
44/100 berhasil
45/100 berhasil
46/100 berhasil
47/100 berhasil
48/100 berhasil
49/100 berhasil
50/100 berhasil
51/100 berhasil
52/100 berhasil
53/100 berhasil
54/100 berhasil
55/100 berhasil
56/100 berhasil
57/100 berhasil
58/100 berhasil
59/100 berhasil
60/100 berhasil
61/100 berhasil
62/100 berhasil
63/100 berhasil
6

In [14]:
url = "https://finance.detik.com/"

response_finance = requests.get(
    url,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("Status code:", response_finance.status_code)
print("Panjang halaman:", len(response_finance.text))

Status code: 200
Panjang halaman: 354383


In [15]:
links_finance = []

for halaman in range(1, 10):
    url = f"https://finance.detik.com/indeks?page={halaman}"

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    print(f"Mengambil halaman {halaman} - Status: {response.status_code}")

    soup_finance = BeautifulSoup(response.text, "html.parser")

    for a in soup_finance.find_all("a", href=True):
        href = urljoin("https://finance.detik.com/", a["href"])

        # Mengambil URL artikel Finance
        if "finance.detik.com" in href and "/d-" in href:
            links_finance.append(href)

    # Hilangkan duplikat
    links_finance = list(dict.fromkeys(links_finance))

    print(f"Total link unik: {len(links_finance)}")

    if len(links_finance) >= 100:
        break

    time.sleep(1)

links_finance = links_finance[:100]

print("\nJumlah link Finance yang dikumpulkan:", len(links_finance))

Mengambil halaman 1 - Status: 200
Total link unik: 16
Mengambil halaman 2 - Status: 200
Total link unik: 35
Mengambil halaman 3 - Status: 200
Total link unik: 55
Mengambil halaman 4 - Status: 200
Total link unik: 74
Mengambil halaman 5 - Status: 200
Total link unik: 91
Mengambil halaman 6 - Status: 200
Total link unik: 110

Jumlah link Finance yang dikumpulkan: 100


In [16]:
data_finance = []

for i, url in enumerate(links_finance, start=1):
    try:
        downloaded = trafilatura.fetch_url(url)
        isi = trafilatura.extract(downloaded)

        if isi:
            data_finance.append({
                "isi_berita": isi,
                "label": "finance"
            })

            print(f"{i}/100 berhasil")
        else:
            print(f"{i}/100 gagal mengambil isi")

    except Exception as e:
        print(f"{i}/100 error: {e}")

    time.sleep(1)

print("\nTotal berita Finance berhasil:", len(data_finance))

1/100 berhasil
2/100 berhasil
3/100 berhasil
4/100 berhasil
5/100 berhasil
6/100 berhasil
7/100 berhasil
8/100 berhasil
9/100 berhasil
10/100 berhasil
11/100 berhasil
12/100 berhasil
13/100 berhasil
14/100 berhasil
15/100 berhasil
16/100 berhasil
17/100 berhasil
18/100 berhasil
19/100 berhasil
20/100 berhasil
21/100 berhasil
22/100 berhasil
23/100 berhasil
24/100 berhasil
25/100 berhasil
26/100 berhasil
27/100 berhasil
28/100 berhasil
29/100 berhasil
30/100 berhasil
31/100 berhasil
32/100 berhasil
33/100 berhasil
34/100 berhasil
35/100 berhasil
36/100 berhasil
37/100 berhasil
38/100 berhasil
39/100 berhasil
40/100 berhasil
41/100 berhasil
42/100 berhasil
43/100 berhasil
44/100 berhasil
45/100 berhasil
46/100 berhasil
47/100 berhasil
48/100 berhasil
49/100 berhasil
50/100 berhasil
51/100 berhasil
52/100 berhasil
53/100 berhasil
54/100 berhasil
55/100 berhasil
56/100 berhasil
57/100 berhasil
58/100 berhasil
59/100 berhasil
60/100 berhasil
61/100 berhasil
62/100 berhasil
63/100 berhasil
6

In [17]:
# Gabungkan data Sport dan Finance
data_semua = data_sport + data_finance

# Buat DataFrame
df = pd.DataFrame(data_semua)

# Tambahkan ID dari 1 sampai 200
df.insert(0, "id", range(1, len(df) + 1))

# Tampilkan informasi dataset
print("Jumlah total data:", len(df))
print("\nJumlah berdasarkan label:")
print(df["label"].value_counts())

# Tampilkan 5 data pertama
display(df.head())

Jumlah total data: 200

Jumlah berdasarkan label:
label
sport      100
finance    100
Name: count, dtype: int64


,id,isi_berita,label
0,1,Ciamis - Sebanyak 549 siswa dari 40 SD mengiku...,sport
1,2,"Dalam dua seri terakhir MotoGP 2027, Marc Marq...",sport
2,3,"Alwi Farhan, Moh Zaki Ubaidillah dan Muhamad Y...",sport
3,4,Di tengah viral insiden cepirit pada ajang Hyr...,sport
4,5,Dejan Fedinansyah/Felisha Alberta Nathaniel Pa...,sport


In [22]:
df.to_csv(
    "dataset_detik.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("Dataset berhasil disimpan sebagai dataset_detik.csv")

Dataset berhasil disimpan sebagai dataset_detik.csv


In [19]:
print("Jumlah baris:", len(df))
print("Jumlah kolom:", len(df.columns))
print("\nData kosong:")
print(df.isnull().sum())

print("\nDistribusi label:")
print(df["label"].value_counts())

Jumlah baris: 200
Jumlah kolom: 3

Data kosong:
id            0
isi_berita    0
label         0
dtype: int64

Distribusi label:
label
sport      100
finance    100
Name: count, dtype: int64
